# Day 3, morning: deciding what to deploy, and streaming

The same kind of **workbench** as days 1 and 2: you write no code here. Each section calls the package and draws
what comes out. The figures follow lecture L5.

| You implement | In | L5 slides | Sections here |
|---|---|---|---|
| `local_agreement` | `speech/streaming_sim.py` | "LocalAgreement-$n$" | 3, 5 |
| `simulate_streaming` (the clock) | `speech/streaming_sim.py` | "`simulate_streaming`: the clock" | 3, 5 |
| `stream_native` | `speech/streaming_sim.py` | "Native streaming: when is a word final?" | 4, 5 |

The three functions are written except 4 short holes (1 in `local_agreement`, 2 in `simulate_streaming`, 1 in
`stream_native`): every `raise NotImplementedError("TODO(d3): ...")` marks a hole and says what to write there.

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:15 | Sections 1–2 (provided): your laptop, 4 candidates on the same 8 recordings, the wait you predict | about 2 min, plus the first download of the models |
| 0:15–0:40 | `local_agreement` and the clock (3 holes); section 3 on a mock; `pytest -m d3 -k "agreement or clock or model"` | seconds |
| 0:40–0:55 | `stream_native` (1 hole); section 4; `pytest -m d3` | seconds |
| 0:55–1:05 | Section 5: one utterance, three systems | 1 min |
| 1:05–1:20 | Graded run in a terminal (below), with nothing else heavy running; start drafting `DECISIONS.md` | about 6 min |
| 1:20–1:35 | Section 6: your graded rows | seconds |
| 1:35–2:00 | Decide: your deployment choice, budget and predicted wait in `DECISIONS.md` and `results/pareto.yaml` | |

The graded run (Whisper tiny and wav2vec2 re-decoded with LocalAgreement-2 at steps 0.5, 1 and 2 s, and the
reference streaming transducer, on the same 20 LibriSpeech utterances):
```
export DSP_MILESTONE=d3          # PowerShell: $env:DSP_MILESTONE = "d3"
python -m speech.streaming_sim
```
Run the default command above first (7 rows); then append extra systems with `--models <id or demo backend>`,
for example your day-2 decoder registered in `my_backends.py` (lecture L6). Data and models:
`python scripts/fetch_data.py` (it skips what you already have; the reference transducer of the demo is new on
19 November, about 100 MB).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root
DATA = Path(os.environ.get("DSP_DATA", ROOT / "data"))
warnings.filterwarnings("ignore")  # transformers and torch are verbose
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import jiwer
import pandas as pd
from IPython.display import display

from asr_demo import backends
from asr_demo.metrics import run_session_offline
from asr_demo.replay import export_librispeech, replay
from common.bench import hardware_info, load_records
from speech.data import load_librispeech, wer
from speech.streaming_sim import (_mock, commit_lags, load_system, local_agreement, normalise_committed,
                                  reference_word_ends, simulate_streaming, stream_native)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
REFERENCE = "reference/fastconformer-114M-int8"
WORD_ENDS = ROOT / "results/streaming_word_ends.json"   # the cache of the graded run: aligned once


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")

## 1. Your laptop is the demo machine

A latency measured on another machine is not on your front (L5, "Merge every system on the same axes"). The
provided `replay` feeds recordings to demo backends exactly as the browser does (100 ms chunks, then release)
and measures the four numbers of L5, "Push-to-talk: four numbers describe one request". Here: 8 LibriSpeech
utterances, written once to `results/librispeech_wavs/`, through 4 candidates: two offline models from day 2,
the reference streaming transducer, and a smaller transducer. The first use of each model loads it (and
downloads it once).

In [ ]:
print({k: v for k, v in hardware_info().items() if v})
CANDIDATES = ["hf/whisper-tiny.en", "hf/wav2vec2-base-960h", REFERENCE, "reference/zipformer-20M-int8"]
wavs = export_librispeech(DATA / "LibriSpeech", ROOT / "results/librispeech_wavs", n=8)
budget = replay(wavs, CANDIDATES, jsonl=None, limit=8)
if len(budget):
    cols = ["backend", "streaming", "audio_s", "first_partial_ms", "final_latency_ms", "rtf", "wer"]
    per_backend = budget.groupby("backend", sort=False).agg(
        streaming=("streaming", "first"), rtf=("rtf", "median"), wait_ms=("final_latency_ms", "median"),
        first_partial_ms=("first_partial_ms", "median"), wer=("wer", "mean"))
    display(per_backend.round(3))

## 2. Predict the wait before you measure it

L5, "Predict the wait": an **offline** or **re-decoding** model runs its final pass on the buffer at release, so
wait grows roughly linearly with duration (median $\mathrm{wait}/\mathrm{dur}$); a native **streaming** model has
only its last chunk and a flush left, so the wait is roughly constant. Each point below is one recording of section 1;
the lines are those two predictions, with the median wait rate (offline / re-decoding) or median wait (native streaming).

In [ ]:
def _is_native(name):
    b = backends._CACHE.get(name)
    if b is not None:
        return bool(b.streaming and not hasattr(b, "inner"))
    return bool(budget.loc[budget.backend == name, "streaming"].iloc[0])


if "budget" in globals() and len(budget):
    fig, ax = plt.subplots(figsize=(7, 4))
    d = np.linspace(0, budget.audio_s.max() * 1.1, 50)
    rates = (budget.final_latency_ms / budget.audio_s.replace(0, np.nan)).groupby(budget.backend).median()
    for (name, g), color in zip(budget.groupby("backend", sort=False), plt.cm.tab10.colors):
        ax.plot(g.audio_s, g.final_latency_ms, "o", color=color, label=name)
        native = _is_native(name)
        pred = np.full_like(d, g.final_latency_ms.median()) if native else rates[name] * d
        ax.plot(d, pred, "--", color=color, lw=1)
    ax.set(xlabel="utterance duration (s)", ylabel="wait after release (ms)", yscale="log",
           title="wait after release: measured (points), predicted (lines)")
    ax.legend(fontsize=7)
    fig.tight_layout()
    for name, r in per_backend.iterrows():
        native = _is_native(name)
        rate = rates[name]
        how = "constant" if native else f"{rate:.0f} ms/s x duration"
        p5 = r.wait_ms if native else rate * 5
        print(f"{name:36s} predicted wait for a 5 s request: {p5:7.0f} ms  ({how})")

**Questions.** Do the points of each backend follow its line? Whisper's do not: its short requests wait longer
and its long ones less than RTF × duration predicts. Why (L5, "The cost of re-decoding")? Which prediction would
you use for it instead? Set your own budget for a push-to-talk assistant: which wait after release is acceptable,
and which WER? Which candidates meet it on *this* laptop for a 5 s request, and for a 15 s one? Why does the
smaller transducer have such a high WER on these recordings (listen to the start of its outputs; lecture L6
comes back to it)?

## 3. LocalAgreement and the clock, on a mock recogniser

An offline model re-decodes the buffer every `step_s` seconds; a word is committed when the last
$n_{\mathrm{agree}}$ hypotheses agree on it (your `local_agreement`), and the model is busy for the compute
time of each call (your two lines of the clock in `simulate_streaming`). The mock of L5, "What the simulator
shows": 40 words, the last ones of each hypothesis are unstable. The figure gives, for each committed word, the
time it ended in the audio and the time it was committed: the vertical distance to the diagonal is the
commit lag. A model with RTF 0.5 falls behind: its points drift away from the diagonal.

In [ ]:
try:
    print("local_agreement:", local_agreement([["the", "cat", "sad"], ["the", "cat", "sat", "on"]], 2))
except NotImplementedError as e:
    not_yet(e)
rng = np.random.default_rng(0)
mock_words = [f"w{i}" for i in range(40)]
mock_ends = np.cumsum(rng.uniform(0.25, 0.6, 40)) + 0.3
mock_audio = np.zeros(int((mock_ends[-1] + 0.5) * 16000), np.float32)
runs = {}
try:
    for label, cost in (("fast, RTF 0.05", lambda d: 0.05 * d + 0.02), ("slow, RTF 0.5", lambda d: 0.5 * d + 0.1)):
        for step in (0.5, 2.0):
            runs[(label, step)] = simulate_streaming(mock_audio, 16000, _mock(mock_words, mock_ends, 16000, np.random.default_rng(1)),
                                                     step_s=step, cost_model=cost)
except NotImplementedError as e:
    not_yet(e)
if runs:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    index = {w: i for i, w in enumerate(mock_words)}
    rows = []
    for (label, step), r in runs.items():
        pairs = [(mock_ends[index[w]], c) for w, c in zip(r["words"], r["commit_times"]) if w in index]
        ax.plot(*zip(*pairs), ".-", lw=0.8, label=f"{label}, step {step} s")
        lags = commit_lags(r["words"], r["commit_times"], mock_words, mock_ends)
        rows.append(dict(model=label, step_s=step, updates=r["updates"], lag_p50=np.median(lags),
                         finish_delay_s=r["finish_delay_s"], behind=r["behind"]))
    ax.plot([0, mock_ends[-1]], [0, mock_ends[-1]], "k:", lw=1, label="no lag")
    ax.set(xlabel="word ends in the audio (s)", ylabel="word committed (s)", title="mock recogniser, 40 words")
    ax.legend(fontsize=8)
    fig.tight_layout()
    display(pd.DataFrame(rows).round(2))

## 4. Native streaming: a transducer commits a word when the next one starts

The reference transducer of the demo extends its hypothesis every chunk (100 ms), but its last word can still
grow ("the" → "there"). First, its partial results on one LibriSpeech utterance, as the demo shows them; then
your `stream_native`, which commits every word but the last one. The end time of each reference word comes from a
CTC forced alignment of the transcript (L5, "Commit lag: when did each word end?"), computed once and cached.

In [ ]:
utt = load_librispeech(DATA / "LibriSpeech", n=1)[0]
try:
    ref_words, ref_ends = reference_word_ends([utt], cache=WORD_ENDS)[utt["id"]]
    print("reference:", utt["text"].lower())
    print("word ends (s):", ", ".join(f"{w} {e:.2f}" for w, e in list(zip(ref_words, ref_ends))[:8]), "...")
except Exception as e:
    ref_words = ref_ends = None
    print(f"Forced alignment unavailable ({type(e).__name__}: {e}); skipping word end times.")
transducer = None
try:
    transducer = backends.get(REFERENCE)
except backends.BackendUnavailable as e:
    print(e)
if transducer is not None:
    m = run_session_offline(transducer, utt["audio"], utt["sr"], utt["text"], keep_partials=True)
    for t, text in m["partials"][::max(1, len(m["partials"]) // 8)]:
        print(f"  {t:5.1f} s  {text}")
    print(f"final: {m['text']}   (WER {m['wer']:.3f})")
    try:
        r = stream_native(utt["audio"], utt["sr"], transducer, chunk_s=0.1)
        if ref_ends is not None:
            w, c = normalise_committed(r["words"], r["commit_times"])
            lags = commit_lags(w, c, ref_words, ref_ends)
            lag_txt = f"lag p50 {np.median(lags):.2f} s, " if len(lags) else ""
        else:
            lag_txt = ""
        print(f"stream_native: {len(r['words'])} words committed, {lag_txt}"
              f"finish delay {1000 * r['finish_delay_s']:.0f} ms, {r['updates']} calls, behind: {r['behind']}")
    except NotImplementedError as e:
        not_yet(e)

## 5. One utterance, three systems, on the same clock

Whisper tiny re-decoded every second, wav2vec2 re-decoded every 0.5 s (both with LocalAgreement-2), and the
transducer, natively. These are the configurations of the graded run, on its first utterance, with the real
compute time of this laptop. Points: each correct word, its end in the audio against its commit time.

In [ ]:
if "utt" not in globals():
    utt = load_librispeech(DATA / "LibriSpeech", n=1)[0]
if "ref_ends" not in globals() or ref_ends is None:
    try:
        ref_words, ref_ends = reference_word_ends([utt], cache=WORD_ENDS)[utt["id"]]
    except Exception as e:
        ref_words = ref_ends = None
        print(f"Forced alignment unavailable ({type(e).__name__}: {e}); skipping commit-lag plot.")
systems = [("hf/whisper-tiny.en", 1.0), ("hf/wav2vec2-base-960h", 0.5), (REFERENCE, 0.1)]
fig, ax = plt.subplots(figsize=(7, 4.5))
rows = []
for name, step in systems:
    try:
        kind, system = load_system(name)
        r = (stream_native(utt["audio"], utt["sr"], system, chunk_s=step) if kind == "native"
             else simulate_streaming(utt["audio"], utt["sr"], system, step_s=step))
    except backends.BackendUnavailable as e:
        print(e)
        continue
    except NotImplementedError as e:
        not_yet(e)
        break
    if ref_ends is not None:
        w, c = normalise_committed(r["words"], r["commit_times"])
        out = jiwer.process_words(" ".join(ref_words), " ".join(w) or "<e>")
        pairs = [(ref_ends[i], c[j]) for ch in out.alignments[0] if ch.type == "equal"
                 for i, j in zip(range(ch.ref_start_idx, ch.ref_end_idx), range(ch.hyp_start_idx, ch.hyp_end_idx))]
        if pairs:
            ax.plot(*zip(*pairs), "o-", ms=3, lw=0.8, label=f"{name}, {'chunk' if kind == 'native' else 'step'} {step} s")
        lags = np.array([b - a for a, b in pairs])
        lag_p50 = np.median(lags) if len(lags) else None
    else:
        lag_p50 = None
    rows.append(dict(system=name, kind=kind, step_s=step, wer=wer([utt["text"]], [r["text"]]),
                     lag_p50=lag_p50, finish_delay_ms=1000 * r["finish_delay_s"],
                     calls=r["updates"], behind=r["behind"]))
dur = len(utt["audio"]) / utt["sr"]
ax.plot([0, dur], [0, dur], "k:", lw=1, label="no lag")
ax.axvline(dur, color="grey", lw=0.8)
ax.set(xlabel="word ends in the audio (s)", ylabel="word committed (s)", title=f"{utt['id']}, {dur:.1f} s")
ax.legend(fontsize=7)
fig.tight_layout()
if rows:
    display(pd.DataFrame(rows).round(3))

**Questions.** Whisper pads every input to 30 s: what does that do to the cost of one update, whatever the
buffer length, and so to the smallest step it can keep up with? Why is the finish delay of a re-decoded model
about one decoding of the whole utterance, and that of the transducer a few tens of milliseconds?

## 6. Your graded rows, as a front

After the graded run (top of the notebook), this cell reads the `asr_streaming` rows of
`results/results.jsonl`: WER of the committed text against the median commit lag (one point per system and
step), and the wait after release (`finish_p50`) against the compute per second of audio (`rtf`).

In [ ]:
res = ROOT / "results/results.jsonl"
df = load_records(res) if res.exists() else None
if df is not None and "task" in df:
    df = df[df.task == "asr_streaming"].copy()
    if {"cfg_policy", "finish_p50"}.issubset(df.columns):
        df = df.dropna(subset=["cfg_policy", "finish_p50"])
    else:
        df = df.iloc[0:0]
else:
    df = None
if df is None or len(df) == 0:
    print("No current 'asr_streaming' rows yet (with cfg_policy and finish_p50): run the graded command above, then re-run this cell.")
else:
    df = df.drop_duplicates(["model", "cfg_policy", "cfg_step_s"], keep="last").copy()
    df["label"] = df.model.str.split("/").str[-1] + " " + df.cfg_step_s.astype(str) + " s"
    show = ["model", "cfg_policy", "cfg_step_s", "wer_stream", "wer_offline", "lag_p50", "lag_p90", "finish_p50",
            "rtf", "behind_frac"]
    display(df[[c for c in show if c in df]].round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for name, g in df.groupby("model", sort=False):
        axes[0].plot(g.lag_p50, 100 * g.wer_stream, "o-", label=name)
        axes[1].plot(g.rtf, 1000 * g.finish_p50, "o", label=name)
        for _, r in g.iterrows():
            axes[0].annotate(f"{r.cfg_step_s:g}", (r.lag_p50, 100 * r.wer_stream), fontsize=7, xytext=(3, 3),
                             textcoords="offset points")
    axes[0].set(xlabel="median commit lag (s)", ylabel="WER of the committed text (%)", title="streaming front")
    axes[1].set(xlabel="RTF (compute / audio)", ylabel="median wait after release (ms)", xscale="log",
                yscale="log", title="cost")
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    (ROOT / "results/figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(ROOT / "results/figures/streaming_pareto.png", dpi=130)

## Before you leave: decide

Write the `DECISIONS.md` entry tagged d3 **before** the afternoon's demo (L5, "Lab, this morning"): the system you
would deploy behind the push-to-talk button, your budget (wait after release, WER), and what you predict the
demo will show for it (first partial, wait after release, WER on your own voice). In `results/pareto.yaml`, the
columns of your main front are columns of `results/results.jsonl`, for example `wer_stream` (quality, `min`) and
`lag_p50` or `finish_p50` (cost, `min`) for the streaming rows; the budget is in the unit of the cost column.